# Small-state scan sizing and data-quality checks
Source: USDA FNS SNAP public ArcGIS snapshot; retained ATP source assertions and the pre-run serving catalog.
SNAP grocery-format counts size the workload. They exclude non-SNAP shops and precede brand-level exclusions, so they are not unique-store census counts. No cross-source count subtraction establishes missing stores. The initial crawl covers retained public branch URLs; source gaps remain explicit.


In [1]:
import json
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bord_food_api").is_dir())
sys.path.insert(0, str(root))
from bord_food_api.state_campaigns import group_states, rank_states

here = root / "docs/investigations/small-states-20261020"
reference = json.loads((here / "snap-groups.json").read_text())
retained = json.loads((here / "retained-counts.json").read_text())
assert reference["group_total_matches_reference"] and reference["state_count"] == 50
assert len({(g["State"], g["Store_Type"]) for g in reference["groups"]}) == len(reference["groups"])
assert all(type(g["store_count"]) is int and g["store_count"] >= 0 for g in reference["groups"])
ranked = rank_states(
    reference, roster_counts=retained["roster_counts"], catalog_counts=retained["catalog_counts"]
)
assert len(ranked) == 50 and len({r["state"] for r in ranked}) == 50
assert ranked == json.loads((here / "ranking.json").read_text())["states"]
print(
    {
        "grain": "one state; grocery-format SNAP records, not a census",
        "groups": len(reference["groups"]),
        "all_format_reference_rows": reference["reference_rows_all_formats"],
        "states": len(ranked),
        "source_clock": reference["source_clock"],
    }
)

{'grain': 'one state; grocery-format SNAP records, not a census', 'groups': 360, 'all_format_reference_rows': 248622, 'states': 50, 'source_clock': 1791394759439}


In [2]:
batches = group_states(ranked, count=12, capacity=650, max_states=3)
plan = json.loads((here / "plan.json").read_text())
assert batches == plan["batches"]
selected = [s for b in batches for s in b["states"]]
assert len(selected) == len(set(selected)) == 12 and "IA" not in selected and "DC" not in selected
assert all(b["snap_reference_workload"] <= 650 and len(b["states"]) <= 3 for b in batches)
for b in batches:
    print(b)
print(
    {
        "selected_states": len(selected),
        "retained_source_assertions": sum(retained["roster_counts"][s] for s in selected),
        "not_performed_by_notebook": "No network, source imports, model calls, crawler activation or store publication",
    }
)

{'batch': 1, 'states': ['WY', 'AK', 'VT'], 'snap_reference_workload': 377}
{'batch': 2, 'states': ['ND', 'DE', 'SD'], 'snap_reference_workload': 603}
{'batch': 3, 'states': ['MT', 'RI'], 'snap_reference_workload': 501}
{'batch': 4, 'states': ['NH', 'HI'], 'snap_reference_workload': 539}
{'batch': 5, 'states': ['ID', 'WV'], 'snap_reference_workload': 587}
{'selected_states': 12, 'retained_source_assertions': 947, 'not_performed_by_notebook': 'No network, source imports, model calls, crawler activation or store publication'}
